# Desarrollo de Aplicaciones Web (DAB-2201)
## Sesión 21: Fundamentos de Persistencia: SQLite / PostgreSQL y Drivers
**Docente Titular:** MCC. Iván Márquez Larios (`ijmarquezl`)
**Repositorio Maestro:** `https://github.com/ijmarquezl/tecnm-desarrollo-web`

---
### Protocolo de la Sesión
1. **Fase 1 (El Artesano):** Resolver a mano: *Conexión manual y ejecución de sentencias SQL*.
2. **Fase 2 (El Científico):** Definir contrato C-R-E-O: *Patrón Singleton de pool de conexiones asíncronas* y formular prompt.
3. **Fase 3 (El Auditor):** Someter el servidor al Test Harness: *Pruebas de recuperación ante desconexión de DB*.
4. **Entrega:** Documentar en `auditorias/auditoria_sesion21.md`.

### Paso 0: Inicialización del Entorno (FastAPI y PHP 8+)

In [1]:
import os
print('⏳ Instalando FastAPI, Uvicorn y HTTPX para pruebas...')
!pip install -q fastapi uvicorn httpx pydantic passlib python-jose[cryptography] sqlalchemy alembic
if os.system('which php > /dev/null 2>&1') != 0:
    print('⏳ Instalando PHP CLI...')
    !apt-get update -qq && apt-get install -y -qq php-cli > /dev/null 2>&1
print('✅ Entorno Web listo:')
!python3 --version
!php -v | head -n 1

⏳ Instalando FastAPI, Uvicorn y HTTPX para pruebas...
⏳ Instalando PHP CLI...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
✅ Entorno Web listo:
Python 3.13.18
PHP 8.3.6 (cli) (built: Nov 18 2026 23:42:07) (NTS)


---
## SECCIÓN A: TRACK PYTHON / FASTAPI
### Fase 1: El Artesano (15 min - Sin IA)
**Reto:** Conexión manual y ejecución de sentencias SQL

In [3]:
%%writefile app_artesano.py
import sqlite3
from fastapi import FastAPI, HTTPException, status
from pydantic import BaseModel

app = FastAPI(title="API Artesano")

ARCHIVO_DB = "datos_s21.db"


class EntradaDato(BaseModel):
    dato: str


def preparar_tabla():
    con = sqlite3.connect(ARCHIVO_DB)
    cur = con.cursor()
    cur.execute(
        "CREATE TABLE IF NOT EXISTS registros (id INTEGER PRIMARY KEY AUTOINCREMENT, dato TEXT)"
    )
    con.commit()
    con.close()


preparar_tabla()


@app.post("/api/recurso")
def procesar_recurso(entrada: EntradaDato):
    texto = entrada.dato.strip()
    if not texto:
        raise HTTPException(
            status_code=status.HTTP_400_BAD_REQUEST, detail="Dato vacio"
        )

    try:
        con = sqlite3.connect(ARCHIVO_DB, timeout=2.0)
        cur = con.cursor()
        cur.execute("INSERT INTO registros (dato) VALUES (?)", (texto,))
        nuevo_id = cur.lastrowid
        con.commit()
        con.close()
    except sqlite3.Error as err:
        raise HTTPException(
            status_code=status.HTTP_503_SERVICE_UNAVAILABLE,
            detail=f"Fallo en BD: {err}",
        )

    return {"status": "success", "id": nuevo_id, "mensaje": f"Procesado: {texto}"}


Overwriting app_artesano.py


### Fase 2: El Científico (Prompting C-R-E-O)
Pega aquí la implementación optimizada propuesta por la IA:

In [4]:
%%writefile app_ia.py
import asyncio
import sqlite3
from contextlib import asynccontextmanager
from fastapi import FastAPI, HTTPException, status
from pydantic import BaseModel

DB_FILE = "datos_ia_s21.db"


class EntradaDato(BaseModel):
    dato: str


class PoolConexionesAsync:
    _instancia = None
    _lock_instancia = asyncio.Lock()

    def __init__(self, ruta_db: str, tamano: int = 4):
        self.ruta_db = ruta_db
        self.tamano = tamano
        self._disponibles: asyncio.Queue[sqlite3.Connection] = asyncio.Queue()
        self._lock_pool = asyncio.Lock()
        self._cerrado = False

    @classmethod
    async def obtener_instancia(
        cls, ruta_db: str = DB_FILE, tamano: int = 4
    ) -> "PoolConexionesAsync":
        if cls._instancia is None:
            async with cls._lock_instancia:
                if cls._instancia is None:
                    pool = cls(ruta_db=ruta_db, tamano=tamano)
                    await pool.inicializar()
                    cls._instancia = pool
        return cls._instancia

    async def inicializar(self) -> None:
        async with self._lock_pool:
            self._cerrado = False
            while not self._disponibles.empty():
                conn = self._disponibles.get_nowait()
                try:
                    conn.close()
                except sqlite3.Error:
                    pass

            def _crear_y_preparar():
                con = sqlite3.connect(self.ruta_db, check_same_thread=False, timeout=2.0)
                con.execute(
                    "CREATE TABLE IF NOT EXISTS registros (id INTEGER PRIMARY KEY AUTOINCREMENT, dato TEXT)"
                )
                con.commit()
                return con

            for _ in range(self.tamano):
                con = await asyncio.to_thread(_crear_y_preparar)
                await self._disponibles.put(con)

    async def adquirir(self) -> sqlite3.Connection:
        if self._cerrado:
            raise sqlite3.OperationalError("Pool cerrado")
        return await self._disponibles.get()

    async def liberar(self, con: sqlite3.Connection | None) -> None:
        if con is None:
            return
        if self._cerrado:
            try:
                con.close()
            except sqlite3.Error:
                pass
            return
        await self._disponibles.put(con)

    async def cerrar(self) -> None:
        async with self._lock_pool:
            self._cerrado = True
            while not self._disponibles.empty():
                con = self._disponibles.get_nowait()
                try:
                    con.close()
                except sqlite3.Error:
                    pass
        PoolConexionesAsync._instancia = None


@asynccontextmanager
async def lifespan(app: FastAPI):
    pool = await PoolConexionesAsync.obtener_instancia(DB_FILE, tamano=4)
    app.state.pool = pool
    try:
        yield
    finally:
        await pool.cerrar()


app = FastAPI(title="API IA Pool Async", lifespan=lifespan)


@app.post("/api/recurso")
async def procesar_recurso(entrada: EntradaDato):
    texto = entrada.dato.strip()
    if not texto:
        raise HTTPException(
            status_code=status.HTTP_400_BAD_REQUEST, detail="Dato vacio"
        )

    pool: PoolConexionesAsync = app.state.pool
    con = None
    try:
        con = await pool.adquirir()

        def _insertar():
            cur = con.cursor()
            cur.execute("INSERT INTO registros (dato) VALUES (?)", (texto,))
            nuevo_id = cur.lastrowid
            con.commit()
            cur.close()
            return nuevo_id

        nuevo_id = await asyncio.to_thread(_insertar)
        await pool.liberar(con)
        return {"status": "success", "id": nuevo_id, "mensaje": f"Procesado: {texto}"}
    except (sqlite3.Error, RuntimeError, OSError) as err:
        if con is not None:
            try:
                con.rollback()
                await pool.liberar(con)
            except sqlite3.Error:
                pass
        raise HTTPException(
            status_code=status.HTTP_503_SERVICE_UNAVAILABLE,
            detail=f"Fallo de conexion o BD: {err}",
        )

Writing app_ia.py


### Fase 3: El Auditor (Test Harness HTTP)
Pruebas de servidor: Pruebas de recuperación ante desconexión de DB

In [8]:
import threading
import time
import uvicorn
import httpx
from app_artesano import app

def run_server():
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(1.5)

client = httpx.Client(base_url="http://127.0.0.1:8000")
print("=== AUDITORIA CRITICA WEB: SESION 21 ===\n")
r = client.post("/api/recurso", json={"dato": "Prueba Test Harness"})
print(f"Respuesta Servidor: HTTP {r.status_code} - {r.json()}")
assert r.status_code == 200
print("PASO ✅ - Test Harness HTTP ejecutado con exito.")


ERROR:    [Errno 98] error while attempting to bind on address ('127.0.0.1', 8000): address already in use


=== AUDITORIA CRITICA WEB: SESION 21 ===

Respuesta Servidor: HTTP 200 - {'status': 'success', 'id': 6, 'mensaje': 'Procesado: Prueba Test Harness'}
PASO ✅ - Test Harness HTTP ejecutado con exito.
